# Streaming RAG with LangGraph
## Watch the Graph Think — Node-by-Node Streaming Output
⏱ ~10 min

**Streaming RAG** combines Retrieval-Augmented Generation with real-time node-by-node execution visibility. Instead of waiting for a complete answer, you watch each stage of the pipeline fire as it finishes — retrieval, fallback decisions, and generation all surfaced as they happen.

By the end of this session you will understand *why* streaming matters, *how* LangGraph's `.stream()` differs from `.invoke()`, and *how* to build a conditional RAG pipeline that falls back to web search when local retrieval is sparse.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — Streaming vs batch, and why it matters |
| 2 | **Setup** — Knowledge base, embeddings, and tools |
| 3 | **Graph Architecture** — State, nodes, and conditional edges |
| 4 | **Stream Modes** — `updates` vs `values` vs `messages` |
| 5 | **Conditional Routing** — Local retrieval vs web fallback |
| 6 | **Token-Level Streaming** — Character-by-character LLM output |
| 7 | **Debugging with Streaming** — Using stream output as observability |
| ★ | **Advanced Patterns** — Async streaming, multi-step pipelines |

---

### Prerequisites
- Python 3.10+ (a `.venv` with the requirements already installed)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)

### Key References
> Lewis, P., Perez, E., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS 2020. https://arxiv.org/abs/2005.11401
>
> LangGraph streaming documentation: https://langchain-ai.github.io/langgraph/how-tos/streaming/
>
> LangChain streaming guide: https://python.langchain.com/docs/concepts/streaming/

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "langchain",
            "langchain-openai",
            "langchain-chroma",
            "langchain-community",
            "langchain-text-splitters",
            "langgraph",
            "chromadb",
            "ddgs",
            # "duckduckgo-search",  # legacy package — langchain-community 0.3.31+ requires ddgs instead
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os
from typing import Literal, TypedDict

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY missing or malformed. "
    "Local: add to .env. Colab: add to Secrets panel."
)
print(f"OPENAI_API_KEY present: True (ends ...{key[-4:]})")

---

## Part 1 — Streaming vs Batch: What Changes and Why

---

### The Problem with `.invoke()`

When you call `.invoke()` on a LangGraph pipeline, execution is **completely opaque until the very end**:

```
.invoke()  ──────────────────────────────────────────────────► ANSWER
            retrieve       fallback?       generate
            (running)      (running)       (running)
                ▲               ▲               ▲
                You see nothing until this completes
```

For short pipelines this is fine. But as graphs grow — routing decisions, graders, re-writers, multiple retrieval steps — waiting until `END` becomes painful:
- You cannot tell *which node* is slow
- Users stare at a blank screen
- Debugging requires adding print statements everywhere

---

### `.stream()` Changes Everything

`.stream()` is a Python generator that **yields results as each node finishes**:

```
.stream()  ──────────────────────────────────────────────────► done
             [retrieve]       [generate]
               yield ▲           yield ▲
                     │                 │
              You get this       Then this
              immediately        as soon as it's ready
```

---

### Batch vs Streaming Comparison

| Feature | `.invoke()` | `.stream()` |
|---------|-------------|-------------|
| Return type | Final state dict | Generator of partial updates |
| Visibility | Black box until END | Node-by-node as they complete |
| User experience | Long wait, then full answer | Progressive display |
| Debugging | Add print statements | Stream output IS the debug log |
| Error location | Hard to pinpoint | Immediately obvious which node failed |
| Latency to first output | Full pipeline duration | Time to first node completion |

---

### Streaming Modes in LangGraph

| `stream_mode` | What it yields | Best for |
|---------------|---------------|----------|
| `"updates"` | `{node_name: state_delta}` — only what changed | Live debugging, progressive UI |
| `"values"` | Full accumulated state after each node | When you need the whole picture per step |
| `"debug"` | Low-level execution events (task start/end) | Deep tracing, framework debugging |
| `"messages"` | Token-by-token LLM output chunks | Streaming chat interfaces |

> **This workshop focuses on `"updates"` (most practical) and `"messages"` (for token streaming).**

---

## Part 2 — Setup: Knowledge Base, Embeddings, and Tools

---

### Architecture Overview

```
OFFLINE (index once)
────────────────────────────────────────────────────────────

  SAMPLE_DOCS (5 Python docs)
       │
       ▼
  ┌───────────────────────┐
  │ RecursiveCharSplitter │  chunk_size=300, overlap=30
  └──────────┬────────────┘
             │
             ▼
  ┌───────────────────────┐
  │   OpenAIEmbeddings    │  text-embedding-3-small
  └──────────┬────────────┘
             │
             ▼
  ┌───────────────────────┐
  │  ChromaDB (in-memory) │  collection: "streaming-rag"
  └───────────────────────┘


ONLINE (every query — with streaming)
────────────────────────────────────────────────────────────

  User question
       │
       ▼
  ┌──────────┐  stream yield ①
  │ retrieve │ ──────────────► {"retrieve": {"context": "..."}}
  └─────┬────┘
        │
        ├─ sufficient context ──────────────────────────────┐
        │                                                   │
        └─ sparse context ──► ┌──────────────┐             │
                              │ web_fallback │  yield ②    │
                              └──────┬───────┘             │
                                     │                     │
                                     └─────────────────────┘
                                                 │
                                                 ▼
                                          ┌──────────┐  stream yield ③
                                          │ generate │ ──────────────► {"generate": {"answer": "..."}}
                                          └──────────┘
```

> **Source**: Architecture adapted from Lewis et al. (2020) RAG paper, extended with LangGraph streaming.

In [ ]:
# ─── 2-A: Build the knowledge base ───────────────────────────────────────────
# Five documents covering Python list fundamentals.
# In a real pipeline these would come from a document loader (PDF, web, etc).

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

SAMPLE_DOCS = [
    Document(
        page_content=(
            "Python lists are ordered, mutable sequences. "
            "Create with: my_list = [1, 2, 3]. "
            "Access by index: my_list[0] returns 1. "
            "Negative indexing: my_list[-1] returns the last item."
        ),
        metadata={"source": "python-basics", "topic": "lists-intro"},
    ),
    Document(
        page_content=(
            "List methods: append(x) adds to end, extend(iterable) merges lists, "
            "pop(i) removes by index, sort() sorts in-place, "
            "reverse() flips order, len(list) returns count."
        ),
        metadata={"source": "python-basics", "topic": "list-methods"},
    ),
    Document(
        page_content=(
            "List comprehensions: squares = [x**2 for x in range(10)]. "
            "Filtered: evens = [x for x in range(10) if x % 2 == 0]. "
            "Nested: [[i*j for j in range(3)] for i in range(3)]."
        ),
        metadata={"source": "python-basics", "topic": "comprehensions"},
    ),
    Document(
        page_content=(
            "Slicing lists: my_list[1:3] returns items at index 1 and 2. "
            "my_list[:2] takes the first two. my_list[::2] takes every other item. "
            "Slicing always returns a new list."
        ),
        metadata={"source": "python-basics", "topic": "slicing"},
    ),
    Document(
        page_content=(
            "Python tuples are immutable sequences: t = (1, 2, 3). "
            "Tuple unpacking: a, b, c = (1, 2, 3). "
            "Use tuples for fixed data; use lists when mutation is needed."
        ),
        metadata={"source": "python-basics", "topic": "tuples"},
    ),
]

embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(SAMPLE_DOCS)

vectorstore = Chroma.from_documents(
    chunks,
    embeddings_model,
    collection_name="streaming-rag",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Indexed {len(chunks)} chunks into ChromaDB")
print(f"Topics: {[d.metadata['topic'] for d in SAMPLE_DOCS]}")

In [ ]:
# ─── 2-B: Set up the web search fallback tool ─────────────────────────────────
# DuckDuckGoSearchResults is used when local retrieval is sparse.
# It requires no API key — it queries DuckDuckGo's search results.

from langchain_community.tools import DuckDuckGoSearchResults

# DuckDuckGoSearchResults returns a list of result snippets; max_results controls API cost
# Alternative for simpler use: DuckDuckGoSearchRun() — returns one concatenated string
# web_search_tool = DuckDuckGoSearchRun()  # simpler, no structured snippets
web_search = DuckDuckGoSearchResults(output_format="list", max_results=3)

# Quick sanity check — should return 3 results
test_results = web_search.invoke("Python programming language")
print(f"Web search tool working: {len(test_results)} results returned")
print(f"First result preview: {str(test_results[0])[:120]}...")

---

## Part 3 — Graph Architecture: State, Nodes, and Conditional Edges

---

### LangGraph Core Concepts

A LangGraph graph has three components:

| Component | Role | In this example |
|-----------|------|-----------------|
| **State** (`TypedDict`) | Shared data bag passed between nodes | `question`, `context`, `answer` |
| **Nodes** (functions) | Transform or enrich the state | `retrieve`, `web_fallback`, `generate` |
| **Edges** | Define execution order and branching | `START → retrieve`, conditional router |

### State Design

```python
class RAGState(TypedDict):
    question: str   # set by caller, never mutated
    context:  str   # written by retrieve or web_fallback
    answer:   str   # written by generate
```

Each node receives the **full current state** and returns only the fields it changes.

### Conditional Edge: The Routing Decision

```
retrieve
    │
    ▼
should_use_web(state)
    ├── context is sparse (< 50 chars)  ──►  web_fallback  ──►  generate
    └── context is sufficient           ──────────────────────►  generate
```

The router function returns a **string** matching a node name. LangGraph uses that string to decide which node to call next.

In [ ]:
# ─── 3-A: Define state and node functions ─────────────────────────────────────

from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph


class RAGState(TypedDict):
    question: str
    context: str
    answer: str


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer the question using only the context below. "
            "If the context does not contain enough information, say so.\n\n"
            "Context:\n{context}",
        ),
        ("human", "{question}"),
    ]
)


def retrieve(state: RAGState) -> dict:
    """Node 1: Retrieve top-k chunks from ChromaDB."""
    docs = retriever.invoke(state["question"])
    context = "\n\n".join(d.page_content for d in docs)
    return {"context": context}


def web_fallback(state: RAGState) -> dict:
    """Node 2 (conditional): Search DuckDuckGo when local context is sparse."""
    results = web_search.invoke(state["question"])
    return {"context": str(results)}


def should_use_web(state: RAGState) -> Literal["web_fallback", "generate"]:
    """Router: fire web fallback when local retrieval returned very little."""
    return "web_fallback" if len(state["context"]) < 50 else "generate"


def generate(state: RAGState) -> dict:
    """Node 3: Generate the final answer from retrieved context."""
    response = (prompt | llm).invoke(
        {"context": state["context"], "question": state["question"]}
    )
    return {"answer": response.content}


print("Node functions defined: retrieve, web_fallback, should_use_web, generate")

In [ ]:
# ─── 3-B: Assemble and compile the graph ──────────────────────────────────────

graph_builder = StateGraph(RAGState)

# Register nodes
graph_builder.add_node("retrieve", retrieve)
graph_builder.add_node("web_fallback", web_fallback)
graph_builder.add_node("generate", generate)

# Define edges
graph_builder.add_edge(START, "retrieve")                        # always start with retrieval
graph_builder.add_conditional_edges("retrieve", should_use_web)  # route after retrieval
graph_builder.add_edge("web_fallback", "generate")               # fallback leads to generate
graph_builder.add_edge("generate", END)                          # generation is always last

app = graph_builder.compile()

print("Graph compiled successfully.")
print("Execution paths:")
print("  Path A (local):  START → retrieve → generate → END")
print("  Path B (web):    START → retrieve → web_fallback → generate → END")

---

## Part 4 — Stream Modes: `updates`, `values`, and `messages`

---

LangGraph's `.stream()` accepts a `stream_mode` parameter that controls what gets yielded at each step.

### `stream_mode="updates"` — the delta stream

Yields a dict with **only what changed** after each node:

```python
# After retrieve node:
{"retrieve": {"context": "Python lists are ordered..."}}

# After generate node:
{"generate": {"answer": "A Python list is..."}}
```

### `stream_mode="values"` — the full-state stream

Yields the **complete accumulated state** after each node:

```python
# After retrieve node:
{"question": "What is a list?", "context": "Python lists...", "answer": ""}

# After generate node:
{"question": "What is a list?", "context": "Python lists...", "answer": "A list is..."}
```

### Choosing a mode

- **`updates`**: Building a live UI that shows "Retrieving... Generating..." — you only need the delta.
- **`values`**: Debugging data transformations — you want the whole state at each checkpoint.
- **`messages`**: Streaming chat UI — you need token-by-token LLM output.

In [ ]:
# ─── 4-A: stream_mode="updates" — the delta stream ────────────────────────────
# Each yielded chunk is {node_name: state_delta}.
# This is the most useful mode for live UI and debugging.

question = "What is a Python list and how do you use it?"
initial_state = {"question": question, "context": "", "answer": ""}

print(f"Q: {question}")
print("─" * 60)
print("stream_mode='updates' output:\n")

for chunk in app.stream(initial_state, stream_mode="updates"):
    for node_name, update in chunk.items():
        print(f"[{node_name}] yielded delta:")
        for key, value in update.items():
            preview = str(value)[:120].replace("\n", " ")
            if len(str(value)) > 120:
                print(f"  {key}: {preview}...")
            else:
                print(f"  {key}: {preview}")
        print()

In [ ]:
# ─── 4-B: stream_mode="values" — the full-state stream ───────────────────────
# Each yielded chunk is the COMPLETE state after that node.
# After retrieve: context is populated, answer is still empty.
# After generate: all three fields are filled.

print(f"Q: {question}")
print("─" * 60)
print("stream_mode='values' output:\n")

for i, state_snapshot in enumerate(app.stream(initial_state, stream_mode="values")):
    print(f"Snapshot {i} (after node execution):")
    print(f"  question: {state_snapshot['question'][:60]}")
    ctx = state_snapshot["context"]
    if ctx:
        print(f"  context:  {ctx[:80].replace(chr(10), ' ')}...")
    else:
        print("  context:  (empty)")
    ans = state_snapshot["answer"]
    if ans:
        print(f"  answer:   {ans[:80]}")
    else:
        print("  answer:   (not yet generated)")
    print()

In [ ]:
# ─── 4-C: Compare .invoke() vs .stream() timing side by side ─────────────────
# The timing output makes the streaming advantage concrete.

import time

print("=" * 60)
print("METHOD A: .invoke() — blocks until END, returns final state")
print("=" * 60)
t0 = time.time()
final_state = app.invoke(initial_state)
elapsed = time.time() - t0
print(f"Total wait: {elapsed:.1f}s")
print(f"Returns: dict with keys {list(final_state.keys())}")
print(f"Answer: {final_state['answer'][:100]}...")

print()
print("=" * 60)
print("METHOD B: .stream() — yields as each node finishes")
print("=" * 60)
t0 = time.time()
for chunk in app.stream(initial_state, stream_mode="updates"):
    elapsed_so_far = time.time() - t0
    node_name = list(chunk.keys())[0]
    print(f"  +{elapsed_so_far:.1f}s  [{node_name}] yielded")
print(f"  +{time.time() - t0:.1f}s  complete")

### Exercise 1 — Switch Stream Modes

In the cell below, replace `stream_mode="updates"` with each of the other modes and observe the output:

1. `stream_mode="values"` — how many snapshots are yielded? What changes between them?
2. `stream_mode="debug"` — what extra fields appear in each yielded chunk?

**Goal:** Understand exactly what each mode yields and why `updates` is the default choice for most pipelines.

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────
ex1_question = "How do I sort a Python list?"

# TODO: change STREAM_MODE to "values" or "debug" and observe the difference
STREAM_MODE = "updates"  # try: "values", "debug"

print(f"stream_mode='{STREAM_MODE}'\n")
for chunk in app.stream(
    {"question": ex1_question, "context": "", "answer": ""},
    stream_mode=STREAM_MODE,
):
    print(chunk)
    print()

---

## Part 5 — Conditional Routing: Local Retrieval vs Web Fallback

---

The `should_use_web` router is the decision point that separates in-domain questions (answered from ChromaDB) from out-of-domain questions (answered from DuckDuckGo). Understanding this pattern is essential for building robust RAG systems.

### The Sparse Context Problem

When a user asks about something *not in the knowledge base*, the retriever still returns chunks — they just have low relevance. Without a fallback, the LLM is forced to answer from irrelevant context, leading to hallucinations or unhelpful responses.

The `should_use_web` gate fires when the total retrieved context is very short (< 50 chars) — a proxy for "the retriever found nothing relevant."

### Threshold Calibration

| Threshold | Effect |
|-----------|--------|
| Too low (< 10 chars) | Almost never fires — even bad retrieval passes |
| 50 chars (default) | Fires only when retrieval returns nothing at all |
| 200 chars | Fires for short retrievals — more questions hit web |
| Embedding similarity score | More principled but requires score-threshold retriever |

In [ ]:
# ─── 5-A: Test routing logic with different questions ─────────────────────────
# Some questions will route to local retrieval, others to the web fallback.

QUESTIONS = [
    ("What is a Python list and how do you use it?", "local"),   # in the KB
    ("How do list comprehensions work in Python?", "local"),     # in the KB
    ("Who won the most recent FIFA World Cup?", "web"),          # NOT in KB
    ("What is the capital of France?", "web"),                   # NOT in KB
]

for question, expected_path in QUESTIONS:
    print(f"Q: {question}")
    print(f"   Expected path: {expected_path}")

    nodes_visited = []
    for chunk in app.stream(
        {"question": question, "context": "", "answer": ""},
        stream_mode="updates",
    ):
        node_name = list(chunk.keys())[0]
        nodes_visited.append(node_name)

    actual_path = " → ".join(nodes_visited)
    matched = ("web_fallback" in nodes_visited) == (expected_path == "web")
    status = "CORRECT" if matched else "UNEXPECTED"
    print(f"   Actual path:   {actual_path}  [{status}]")
    print()

In [ ]:
# ─── 5-B: Rebuild the graph with a tunable threshold ─────────────────────────
# Experiment with routing thresholds without rewriting the router each time.


def build_streaming_rag(threshold: int = 50):
    """Build a streaming RAG app with a configurable fallback threshold."""

    def should_use_web_tunable(
        state: RAGState,
    ) -> Literal["web_fallback", "generate"]:
        return "web_fallback" if len(state["context"]) < threshold else "generate"

    g = StateGraph(RAGState)
    g.add_node("retrieve", retrieve)
    g.add_node("web_fallback", web_fallback)
    g.add_node("generate", generate)
    g.add_edge(START, "retrieve")
    g.add_conditional_edges("retrieve", should_use_web_tunable)
    g.add_edge("web_fallback", "generate")
    g.add_edge("generate", END)
    return g.compile()


# Compare: low vs high threshold for the same question
test_question = "How do list comprehensions work in Python?"

for threshold in [10, 50, 500]:
    tunable_app = build_streaming_rag(threshold=threshold)
    path = []
    for chunk in tunable_app.stream(
        {"question": test_question, "context": "", "answer": ""},
        stream_mode="updates",
    ):
        path.append(list(chunk.keys())[0])

    print(f"threshold={threshold:4d}  →  {' → '.join(path)}")

### Exercise 2 — Tune the Fallback Threshold

Modify the `MY_THRESHOLD` value below and record which questions route to web vs local:

1. `MY_THRESHOLD = 200` — does the Python list question now trigger web?
2. `MY_THRESHOLD = 1000` — how many questions hit the web fallback?
3. `MY_THRESHOLD = 0` — what happens? Does the web path ever fire?

**Key insight:** The length threshold is a blunt instrument. A more principled approach uses `search_type="similarity_score_threshold"` on the retriever — the router then checks whether any chunks were returned at all rather than measuring character count.

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────
# TODO: experiment with different threshold values and observe the routing
MY_THRESHOLD = 200  # try: 0, 10, 50, 200, 1000

my_app = build_streaming_rag(threshold=MY_THRESHOLD)

my_questions = [
    "What is a Python list?",
    "How do I sort a list?",
    "Who is the current US president?",
    # TODO: add your own questions here
]

print(f"Testing with threshold={MY_THRESHOLD}\n")
for q in my_questions:
    path = []
    for chunk in my_app.stream(
        {"question": q, "context": "", "answer": ""},
        stream_mode="updates",
    ):
        path.append(list(chunk.keys())[0])
    route = "WEB" if "web_fallback" in path else "LOCAL"
    print(f"  [{route}] {q[:60]}")

---

## Part 6 — Token-Level Streaming: Character-by-Character LLM Output

---

### Two Levels of Streaming

So far we have streamed at the **node level** — we see each node's output as it completes. But the `generate` node itself still blocks while the LLM produces the full answer.

For a chat UI, you want **token-level streaming** — each word appears as the model produces it.

LangGraph supports this via `stream_mode="messages"`, which yields individual `AIMessageChunk` objects:

```
stream_mode="updates"   →  yield after full node completes
stream_mode="messages"  →  yield per LLM token as generated
```

### How `"messages"` Mode Works

Each yielded item is a tuple `(chunk, metadata)` where:
- `chunk` is an `AIMessageChunk` with a `.content` string (one or a few tokens)
- `metadata` contains `{"langgraph_node": "generate", ...}` identifying the source node

In [ ]:
# ─── 6-A: Token streaming with stream_mode="messages" ─────────────────────────
# Yields individual LLM tokens as the model produces them.
# The output prints word-by-word, simulating a real chat interface.

question = "What is a Python list comprehension and when should I use one?"
print(f"Q: {question}")
print("─" * 60)
print("Token-by-token output (stream_mode='messages'):\n")

for msg_chunk, metadata in app.stream(
    {"question": question, "context": "", "answer": ""},
    stream_mode="messages",
):
    # metadata tells us which node this token came from
    node = metadata.get("langgraph_node", "")

    if node == "generate" and hasattr(msg_chunk, "content") and msg_chunk.content:
        # Print each token without newline — creates the streaming effect
        print(msg_chunk.content, end="", flush=True)

print("\n\n[Stream complete]")

In [ ]:
# ─── 6-B: Simulate a real chat UI — node status + token streaming ─────────────
# Real chat UIs show "Retrieving..." then stream the generated answer.
# This pattern runs two passes: updates for status, messages for output.

question = "How do I slice a Python list?"
state_input = {"question": question, "context": "", "answer": ""}

print(f"Q: {question}")
print("─" * 60)

# Pass 1: collect node-level status via updates
for chunk in app.stream(state_input, stream_mode="updates"):
    for node_name, update in chunk.items():
        if node_name == "retrieve":
            n_chars = len(update.get("context", ""))
            print(f"[{node_name}] Retrieved {n_chars} chars of context")
        elif node_name == "web_fallback":
            print(f"[{node_name}] Fetching web results...")
        # skip generate — we stream tokens separately below

print()
print("Streaming answer token by token:")

# Pass 2: stream tokens from the generate node
for msg_chunk, metadata in app.stream(state_input, stream_mode="messages"):
    if (
        metadata.get("langgraph_node") == "generate"
        and hasattr(msg_chunk, "content")
        and msg_chunk.content
    ):
        print(msg_chunk.content, end="", flush=True)

print("\n")

---

## Part 7 — Debugging with Streaming

---

### Streaming as Observability

One of the most underrated benefits of streaming is that the stream output **is** your observability layer. Instead of adding print statements everywhere or setting up an external tracing tool, you can inspect what each node produces directly from the stream.

### Common Debugging Patterns

| Problem | What to check in the stream |
|---------|-----------------------------|
| Wrong answer | Print the `retrieve` node context — is it relevant? |
| Web fallback always fires | Print `retrieve` context length — is it consistently < threshold? |
| Slow pipeline | Use `time.time()` per node to find the bottleneck |
| Model ignores context | Print the full prompt before generation |
| Web results off-topic | Print `web_fallback` context — DuckDuckGo may need a better query |

In [ ]:
# ─── 7-A: Per-node timing via streaming ───────────────────────────────────────
# Use the stream to measure exactly how long each node takes.

import time


def timed_stream(question: str):
    """Run the pipeline and report per-node latency from the stream output."""
    t_start = time.time()
    t_prev = t_start

    for chunk in app.stream(
        {"question": question, "context": "", "answer": ""},
        stream_mode="updates",
    ):
        t_now = time.time()
        for node_name in chunk:
            elapsed = t_now - t_prev
            print(f"  [{node_name}]  {elapsed:.2f}s")
        t_prev = t_now

    total = time.time() - t_start
    print(f"  [TOTAL]  {total:.2f}s")


print("Timing: local retrieval path")
print("─" * 40)
timed_stream("What is a Python list?")

print()
print("Timing: web fallback path")
print("─" * 40)
timed_stream("Who is the CEO of Apple?")

In [ ]:
# ─── 7-B: Full pipeline diagnostic harness ────────────────────────────────────
# Reference implementation you can adapt as your debug harness.


def debug_pipeline(question: str, show_full_context: bool = False):
    """Run the streaming RAG pipeline with full diagnostic output."""
    print(f"{'=' * 60}")
    print(f"QUESTION: {question}")
    print(f"{'=' * 60}")

    final_answer = ""

    for chunk in app.stream(
        {"question": question, "context": "", "answer": ""},
        stream_mode="updates",
    ):
        for node_name, update in chunk.items():
            print(f"\n[{node_name.upper()}]")

            if "context" in update and update["context"]:
                ctx = update["context"]
                print(f"  context length: {len(ctx)} chars")
                if show_full_context:
                    print(f"  context:\n{ctx}")
                else:
                    print(f"  context preview: {ctx[:150].replace(chr(10), ' ')}...")

            if "answer" in update and update["answer"]:
                final_answer = update["answer"]
                print(f"  answer: {final_answer}")

    print(f"\n{'─' * 60}")
    print(f"FINAL ANSWER: {final_answer}")
    print(f"{'─' * 60}\n")


debug_pipeline("How do I sort a list in Python?")
debug_pipeline("What is the population of Tokyo?")

### Exercise 3 — Add a Third Node: Document Grader

Insert a `grade` node between `retrieve` and the routing decision. The grader should:

1. Check whether the retrieved context is relevant to the question
2. Set a `grade` field in the state (`"relevant"` or `"irrelevant"`)
3. The router should use `grade`, not context length, to decide the path

**Hint:** The grader can use a simple keyword-presence heuristic or a full LLM call with structured output.

**Graph after the change:**
```
START → retrieve → grade → [relevant]   → generate    → END
                         → [irrelevant] → web_fallback → generate → END
```

In [ ]:
# ── Exercise 3 Starter ────────────────────────────────────────────────────────
# Add a grade node and update the state + graph to use it.


class GradedRAGState(TypedDict):
    question: str
    context: str
    grade: str      # TODO: "relevant" or "irrelevant"
    answer: str


def grade_documents(state: GradedRAGState) -> dict:
    """TODO: implement a grader that checks whether context is relevant."""
    # Option A (no API cost): check if question keywords appear in context
    # Option B (LLM): ask the model "is this context relevant to the question?"
    question_words = set(state["question"].lower().split())
    context_words = set(state["context"].lower().split())
    overlap = question_words & context_words

    # TODO: adjust this overlap threshold
    grade = "relevant" if len(overlap) >= 2 else "irrelevant"
    return {"grade": grade}


def route_by_grade(state: GradedRAGState) -> Literal["web_fallback", "generate"]:
    """TODO: route based on grade field instead of context length."""
    return "generate" if state["grade"] == "relevant" else "web_fallback"


# TODO: wire grade_documents into the graph between retrieve and the router
# g = StateGraph(GradedRAGState)
# g.add_node("retrieve", ...)
# g.add_node("grade", grade_documents)
# g.add_node("web_fallback", ...)
# g.add_node("generate", ...)
# g.add_edge(START, "retrieve")
# g.add_edge("retrieve", "grade")               # <── new edge
# g.add_conditional_edges("grade", route_by_grade)  # <── updated router
# ...
# graded_app = g.compile()

print("Starter code defined. Complete the graph wiring in the TODO block above.")

---

## Part 8 ★ — Advanced Patterns (Bonus)

---

### Async Streaming

All LangGraph graphs support async execution via `.astream()`. This is the pattern for production web servers (FastAPI, Starlette) where blocking the event loop is unacceptable:

```python
async for chunk in app.astream(initial_state, stream_mode="updates"):
    for node_name, update in chunk.items():
        print(f"[{node_name}] yielded")
```

### Streaming to FastAPI Server-Sent Events

```python
from fastapi import FastAPI
from fastapi.responses import StreamingResponse

api = FastAPI()

@api.get("/chat")
async def chat(question: str):
    async def token_generator():
        async for msg, meta in app.astream(
            {"question": question, "context": "", "answer": ""},
            stream_mode="messages",
        ):
            if meta.get("langgraph_node") == "generate" and msg.content:
                yield f"data: {msg.content}\n\n"
    return StreamingResponse(token_generator(), media_type="text/event-stream")
```

### Multi-Step Streaming Pipelines

Streaming shines most in longer pipelines with multiple steps:

```
START
  → decompose_query      ← break complex question into sub-queries
  → retrieve_subqueries  ← retrieve for each sub-query
  → merge_context        ← deduplicate and rank chunks
  → generate             ← synthesize final answer
  → validate             ← check answer against sources
END
```

With streaming, each step's output surfaces immediately — a 5-node pipeline is no longer a black box.

In [ ]:
# ─── 8-A: Async streaming demonstration ───────────────────────────────────────
# In a Jupyter notebook, asyncio is already running so we can use await directly.


async def run_async_stream(question: str):
    """Demonstrate async streaming — the pattern for production web servers."""
    print(f"Q: {question}")
    print("Async stream output:")

    async for chunk in app.astream(
        {"question": question, "context": "", "answer": ""},
        stream_mode="updates",
    ):
        for node_name, update in chunk.items():
            print(f"  [{node_name}] async yield")
            if "answer" in update and update["answer"]:
                print(f"  answer: {update['answer'][:120]}")


# In a Jupyter notebook, await works directly in cells
await run_async_stream("How do Python tuples differ from lists?")

In [ ]:
# ─── 8-B: Async token streaming — the production chat pattern ─────────────────


async def stream_chat_response(question: str):
    """Token-by-token async streaming — mirrors a production chat UI."""
    print(f"Q: {question}")
    print("A: ", end="", flush=True)

    async for msg_chunk, metadata in app.astream(
        {"question": question, "context": "", "answer": ""},
        stream_mode="messages",
    ):
        if (
            metadata.get("langgraph_node") == "generate"
            and hasattr(msg_chunk, "content")
            and msg_chunk.content
        ):
            print(msg_chunk.content, end="", flush=True)

    print("\n")


await stream_chat_response("What are list comprehensions and when should I use them?")

---

## What's Next?

You now have a complete foundation in streaming RAG with LangGraph. Here's where to go deeper:

### Back to basics
- **Example 1 — Basic Local RAG** ([`../1-basic-local-rag/`](../1-basic-local-rag/)): the same RAG pipeline without streaming — useful for comparison
- **Example 12 — Basic RAG Notebook** ([`../12-basic-rag-notebook/rag_workbook.ipynb`](../12-basic-rag-notebook/rag_workbook.ipynb)): deep dive into embeddings, ChromaDB, text splitting, and the full LCEL pipeline

### Add intelligence to retrieval
- **Example 25 — Adaptive RAG** ([`../25-adaptive-rag/`](../25-adaptive-rag/)): route queries to different retrieval strategies before retrieval even starts
- **Example 17 — Corrective RAG** ([`../17-corrective-rag/`](../17-corrective-rag/)): grade each retrieved document individually, rewrite the query if documents score poorly

### Evaluate what you built
- **Example 16 — RAG Evaluation** ([`../16-rag-eval-notebook/rag_eval_workbook.ipynb`](../16-rag-eval-notebook/rag_eval_workbook.ipynb)): score your pipeline on Faithfulness, Answer Relevance, and Context Recall

### Scale up
- **Example 30 — Agentic RAG** ([`../30-agentic-rag/`](../30-agentic-rag/)): multi-step agentic retrieval where the LLM decides when to search and what to search for

---

### Academic References

> Lewis, P., Perez, E., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS 2020. https://arxiv.org/abs/2005.11401
>
> Gao, Y., Xiong, Y., et al. (2023). *Retrieval-Augmented Generation for Large Language Models: A Survey.* https://arxiv.org/abs/2312.10997
>
> LangGraph streaming documentation: https://langchain-ai.github.io/langgraph/how-tos/streaming/
>
> LangGraph conceptual guide — streaming: https://langchain-ai.github.io/langgraph/concepts/streaming/

---

*Workshop complete. Open example 25 (Adaptive RAG) to see conditional routing taken to its logical conclusion.*

---

## Exercise Answer Key

Use this section as a reference after attempting the exercises yourself. These are sample solutions, not the only correct answers.

### Exercise 1 — Switch Stream Modes

**`stream_mode="values"`**: Yields one snapshot per node — the full accumulated state after each step. For a 2-node pipeline (retrieve + generate), you get 2 snapshots. The difference between consecutive snapshots shows exactly what each node changed.

**`stream_mode="debug"`**: Yields low-level execution events including task start, task end, and checkpoint events. Each event has a `type` field (`"task"`, `"task_result"`, `"checkpoint"`) and a `payload`. Useful for framework-level debugging but too verbose for production.

**Rule of thumb:**
- `updates` for production pipelines and live UIs
- `values` when debugging data transformations step by step
- `debug` only when investigating LangGraph internals or unexpected behavior

In [ ]:
# Answer Key: Exercise 1
# Demonstrates the concrete difference between updates and values modes.

ak1_question = "How do I sort a Python list?"
ak1_state = {"question": ak1_question, "context": "", "answer": ""}

print("=== updates mode — yields only what changed ===")
for chunk in app.stream(ak1_state, stream_mode="updates"):
    for node, delta in chunk.items():
        print(f"  [{node}] changed keys: {list(delta.keys())}")

print()
print("=== values mode — yields full state at every step ===")
for i, snap in enumerate(app.stream(ak1_state, stream_mode="values")):
    ctx_status = "present" if snap["context"] else "empty"
    ans_status = "present" if snap["answer"] else "empty"
    print(f"  snapshot {i}: context={ctx_status}, answer={ans_status}")

### Exercise 2 — Tune the Fallback Threshold

**Expected findings:**
- `threshold=0`: web fallback **never** fires (context length is always ≥ 0 chars)
- `threshold=50` (default): fires only when ChromaDB returns nothing at all
- `threshold=200`: fires for short retrievals — in-KB questions with compact answers may hit web
- `threshold=1000`: almost all questions hit web fallback (most contexts are < 1000 chars)

**Better approach than a length threshold:**

```python
# Use a score-threshold retriever — returns NOTHING when similarity is low
retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.7},
)

# Router checks if ANY docs were returned rather than measuring chars
def should_use_web(state: RAGState) -> str:
    return "web_fallback" if not state["context"] else "generate"
```

### Exercise 3 — Add a Third Node: Document Grader

**What good output looks like:**
- In-domain question: `retrieve → grade → generate` (grade=relevant)
- Out-of-domain question: `retrieve → grade → web_fallback → generate` (grade=irrelevant)

**Common pitfall:** An LLM-based grader adds an extra API call — this roughly doubles latency on the local path. For production, prefer heuristics (keyword overlap, context length, embedding score) and reserve LLM graders for high-stakes pipelines.

**The streaming benefit is visible here:** with a grader node inserted, you now see three yield events before the answer appears — `[retrieve]`, `[grade]`, `[generate]` — making the routing decision visible in real time.

In [ ]:
# Answer Key: Exercise 3 — Complete graded RAG graph


class GradedRAGStateFull(TypedDict):
    question: str
    context: str
    grade: str
    answer: str


def retrieve_ak(state: GradedRAGStateFull) -> dict:
    docs = retriever.invoke(state["question"])
    return {"context": "\n\n".join(d.page_content for d in docs)}


def grade_ak(state: GradedRAGStateFull) -> dict:
    # Remove common stopwords before checking overlap
    stopwords = {"what", "is", "a", "the", "how", "do", "i", "in", "of", "and", "to", "for"}
    question_keywords = set(state["question"].lower().split()) - stopwords
    context_words = set(state["context"].lower().split())
    overlap = question_keywords & context_words
    grade = "relevant" if len(overlap) >= 2 else "irrelevant"
    return {"grade": grade}


def route_by_grade_ak(
    state: GradedRAGStateFull,
) -> Literal["web_fallback", "generate"]:
    return "generate" if state["grade"] == "relevant" else "web_fallback"


def web_fallback_ak(state: GradedRAGStateFull) -> dict:
    results = web_search.invoke(state["question"])
    return {"context": str(results)}


def generate_ak(state: GradedRAGStateFull) -> dict:
    response = (prompt | llm).invoke(
        {"context": state["context"], "question": state["question"]}
    )
    return {"answer": response.content}


graded_builder = StateGraph(GradedRAGStateFull)
graded_builder.add_node("retrieve", retrieve_ak)
graded_builder.add_node("grade", grade_ak)
graded_builder.add_node("web_fallback", web_fallback_ak)
graded_builder.add_node("generate", generate_ak)
graded_builder.add_edge(START, "retrieve")
graded_builder.add_edge("retrieve", "grade")              # NEW: grade after retrieve
graded_builder.add_conditional_edges("grade", route_by_grade_ak)  # NEW: route from grade
graded_builder.add_edge("web_fallback", "generate")
graded_builder.add_edge("generate", END)
graded_app = graded_builder.compile()

print("Graded RAG graph compiled. Testing:\n")
test_cases = [
    "What is a Python list?",
    "What is the capital of Australia?",
]
for q in test_cases:
    path = []
    for chunk in graded_app.stream(
        {"question": q, "context": "", "grade": "", "answer": ""},
        stream_mode="updates",
    ):
        path.extend(chunk.keys())
    print(f"  Q: {q[:55]}")
    print(f"     Path: {' → '.join(path)}")